In [24]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import pandas as pd

In [25]:
save_path = "../AutogluonModels/TimeSeriesPredictor"
df = pd.read_csv("../data/processed/datas_cleaned.csv")
df["item_id"] = df["region"] + "_" + df["size_rank"].astype(str)
df["timestamp"] = pd.to_datetime((df["year"] - 543).astype(str))
df.head()

,year,region,size_estabish,value,size_rank,item_id,timestamp
0,2542,Bangkok,<60,"297,061,700",1,Bangkok_1,1999-01-01
1,2544,Bangkok,<60,"316,149,600",1,Bangkok_1,2001-01-01
2,2546,Bangkok,<60,"292,475,000",1,Bangkok_1,2003-01-01
3,2548,Bangkok,<60,"318,711,400",1,Bangkok_1,2005-01-01
4,2550,Bangkok,<60,"443,951,100",1,Bangkok_1,2007-01-01


In [26]:
ts_df = TimeSeriesDataFrame.from_data_frame(
    df[["item_id", "timestamp", "value"]],
    id_column="item_id",
    timestamp_column="timestamp",
)
ts_df.head()

value
item_id   timestamp             
Bangkok_1 1999-01-01 297,061,700
          2001-01-01 316,149,600
          2003-01-01 292,475,000
          2005-01-01 318,711,400
          2007-01-01 443,951,100

In [27]:
predictor = TimeSeriesPredictor(
    prediction_length=2, target="value", eval_metric="MASE", path=save_path
).fit(ts_df, presets="best_quality", time_limit=600)

Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to 'd:\Mini Project\Term Project\AutogluonModels\TimeSeriesPredictor'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          20
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       1.16 GB / 15.73 GB (7.4%)
Disk Space Avail:   636.25 GB / 953.85 GB (66.7%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 'auto',
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 'auto',
 'refit_full': 

In [28]:
predictor.evaluate(ts_df)

Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


{'MASE': -1.6288795846832498}

In [29]:
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")  # จัดรูปแบบตัวเลข
predictions = predictor.predict(ts_df).reset_index()
print(predictions["mean"])

Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


0     2,517,816,535
1     2,539,888,166
2    12,537,882,605
3    12,721,640,980
4    50,297,179,908
5    48,825,687,182
6     7,686,412,031
7     7,644,320,515
8     8,240,081,426
9     8,063,743,683
10   20,428,315,299
11   19,745,903,075
12    4,834,914,691
13    4,859,272,201
14    3,939,926,412
15    3,980,509,408
16    5,251,975,462
17    5,121,495,661
18    2,559,032,163
19    2,705,048,128
20    2,522,007,551
21    2,578,812,579
22    3,343,978,755
23    3,393,418,390
24   16,500,093,523
25   16,235,082,984
26   14,853,795,318
27   14,449,017,147
28   35,043,549,359
29   34,047,197,759
Name: mean, dtype: float64


In [30]:
predictions["year_th"] = predictions["timestamp"].dt.year + 543
predictions[["region", "size_rank"]] = predictions["item_id"].str.rsplit(
    "_", n=1, expand=True
)
predictions["size_rank"] = predictions["size_rank"].astype(int)

In [31]:
actual = df[["item_id", "year", "value", "region", "size_rank"]].copy()
actual["year_th"] = actual["year"]
actual["type"] = "actual"

In [32]:
pred_export = predictions[
    ["item_id", "year_th", "region", "size_rank", "mean", "0.1", "0.9"]
].copy()
pred_export.columns = [
    "item_id",
    "year_th",
    "region",
    "size_rank",
    "value",
    "lower",
    "upper",
]
pred_export["type"] = "forecast"

actual_export = actual[["item_id", "year_th", "region", "size_rank", "value"]].copy()
actual_export["type"] = "actual"

combined = pd.concat([actual_export, pred_export], ignore_index=True)
combined.to_csv("../data/predict/predictions.csv", index=False)
print("Saved predictions.csv")

Saved predictions.csv
